<a href="https://colab.research.google.com/github/madiar-ops/AIpython/blob/main/eleventhlesson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from google.colab import files
from IPython.display import display

print(" Библиотеки успешно загружены")
print(f"   OpenCV версия: {cv2.__version__}")
print(f"   NumPy версия:  {np.__version__}")



def analyze_image(image_path: str, image_label: str) -> dict:



    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(f" Файл '{image_path}' не найден или повреждён.")

    print(f"\n{'='*55}")
    print(f"  {image_label}")
    print(f"{'='*55}")
    print(f" Изображение успешно загружено.")
    print(f"   Размер: {img_bgr.shape[1]} × {img_bgr.shape[0]} пикселей, "
          f"каналов: {img_bgr.shape[2]}")

    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

    sobel_combined = cv2.magnitude(sobel_x, sobel_y)

    sobel_x_disp  = cv2.normalize(np.abs(sobel_x), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    sobel_y_disp  = cv2.normalize(np.abs(sobel_y), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    sobel_comb_disp = cv2.normalize(sobel_combined, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    sobel_x_blur = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y_blur = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    sobel_comb_blur = cv2.magnitude(sobel_x_blur, sobel_y_blur)
    sobel_blur_disp = cv2.normalize(sobel_comb_blur, None, 0, 255,
                                    cv2.NORM_MINMAX).astype(np.uint8)

    edge_pixels_no_blur   = np.sum(sobel_comb_disp > 30)
    edge_pixels_with_blur = np.sum(sobel_blur_disp  > 30)
    noise_reduction_pct   = (1 - edge_pixels_with_blur / max(edge_pixels_no_blur, 1)) * 100

    print(f"   Пикселей-границ без размытия:  {edge_pixels_no_blur:,}")
    print(f"   Пикселей-границ с размытием:   {edge_pixels_with_blur:,}")
    print(f"   Снижение шума:                 {noise_reduction_pct:.1f}%")

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    fig = plt.figure(figsize=(18, 10))
    fig.suptitle(image_label, fontsize=15, fontweight="bold", y=1.01)

    gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.35, wspace=0.3)

    panels = [
        (gs[0, 0], img_rgb,         "Оригинал (RGB)",         "viridis"),
        (gs[0, 1], gray,             "Оттенки серого",          "gray"),
        (gs[0, 2], sobel_x_disp,    "Sobel X\n(вертикальные границы)", "gray"),
        (gs[0, 3], sobel_y_disp,    "Sobel Y\n(горизонтальные границы)", "gray"),
        (gs[1, 0], sobel_comb_disp, "Sobel Combined\n(без размытия)",   "gray"),
        (gs[1, 1], blurred,          "GaussianBlur (5×5)",       "gray"),
        (gs[1, 2], sobel_blur_disp, "Sobel Combined\n(с размытием)",    "gray"),
    ]

    for spec, img_data, title, cmap in panels:
        ax = fig.add_subplot(spec)
        ax.imshow(img_data, cmap=cmap if img_data.ndim == 2 else None)
        ax.set_title(title, fontsize=10)
        ax.axis("off")

    ax_last = fig.add_subplot(gs[1, 3])
    mid_row = gray.shape[0] // 2
    ax_last.plot(sobel_comb_disp[mid_row, :],  label="Без blur",  alpha=0.8, linewidth=1)
    ax_last.plot(sobel_blur_disp[mid_row, :],  label="С blur",    alpha=0.8, linewidth=1)
    ax_last.set_title("Профиль границ\n(средняя строка)", fontsize=10)
    ax_last.set_xlabel("Пиксель X", fontsize=8)
    ax_last.set_ylabel("Интенсивность", fontsize=8)
    ax_last.legend(fontsize=8)
    ax_last.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return {
        "label":             image_label,
        "shape":             img_bgr.shape,
        "edge_no_blur":      edge_pixels_no_blur,
        "edge_with_blur":    edge_pixels_with_blur,
        "noise_reduction":   noise_reduction_pct,
        "sobel_x_mean":      float(np.mean(sobel_x_disp)),
        "sobel_y_mean":      float(np.mean(sobel_y_disp)),
        "sobel_comb_disp":   sobel_comb_disp,
        "sobel_blur_disp":   sobel_blur_disp,
    }

print(" Функция analyze_image() определена.")




print(" Загрузите изображение 1 (Портрет / лицо):")
uploaded = files.upload()
img1_path = list(uploaded.keys())[0]
print(f" Загружен файл: {img1_path}")

results_1 = analyze_image(img1_path, "Изображение 1: Портрет человека")



print(" Загрузите изображение 2 (Здание / улица):")
uploaded = files.upload()
img2_path = list(uploaded.keys())[0]
print(f" Загружен файл: {img2_path}")

results_2 = analyze_image(img2_path, "Изображение 2: Здание / улица")

print(" Загрузите изображение 3 (Предмет крупным планом):")
uploaded = files.upload()
img3_path = list(uploaded.keys())[0]
print(f" Загружен файл: {img3_path}")

results_3 = analyze_image(img3_path, "Изображение 3: Предмет крупным планом")



import pandas as pd

all_results = [results_1, results_2, results_3]

summary = pd.DataFrame([
    {
        "Изображение":          r["label"].split(": ")[1],
        "Размер (px)":          f"{r['shape'][1]}×{r['shape'][0]}",
        "Sobel X (ср.)":        f"{r['sobel_x_mean']:.1f}",
        "Sobel Y (ср.)":        f"{r['sobel_y_mean']:.1f}",
        "Границ без blur":       f"{r['edge_no_blur']:,}",
        "Границ с blur":         f"{r['edge_with_blur']:,}",
        "Снижение шума (%)":     f"{r['noise_reduction']:.1f}",
    }
    for r in all_results
])

print("\n Сводная таблица результатов:")
print(summary.to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Сравнение: Sobel Combined (без blur vs с blur)", fontsize=14, fontweight="bold")

labels_short = ["Портрет", "Здание", "Предмет"]

for col, (r, lbl) in enumerate(zip(all_results, labels_short)):
    axes[0, col].imshow(r["sobel_comb_disp"], cmap="gray")
    axes[0, col].set_title(f"{lbl}\nSobel (без blur)", fontsize=10)
    axes[0, col].axis("off")

    axes[1, col].imshow(r["sobel_blur_disp"], cmap="gray")
    axes[1, col].set_title(f"{lbl}\nSobel (с blur)", fontsize=10)
    axes[1, col].axis("off")

plt.tight_layout()
plt.show()

fig2, ax = plt.subplots(figsize=(8, 4))
noise_vals = [r["noise_reduction"] for r in all_results]
bars = ax.bar(labels_short, noise_vals,
              color=["#4e79a7", "#f28e2b", "#59a14f"], edgecolor="white", width=0.5)
ax.set_title("Снижение числа пикселей-границ после GaussianBlur (%)",
             fontsize=12, fontweight="bold")
ax.set_ylabel("Снижение (%)")
ax.set_ylim(0, max(noise_vals) * 1.3)
for bar, val in zip(bars, noise_vals):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5, f"{val:.1f}%",
            ha="center", va="bottom", fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


